# Malinois MPRA 5: ΔP — the CAV's own prediction of a mutation's effect

Until now the CAV only weighted someone else's prediction: P(bound) of the reference occurrence multiplied motifbreakR's
alleleDiff or deltaSVM. Here the CAV predicts the effect itself: for every motif occurrence that contains a variant,
the 600 bp window is embedded again **with the alternative base**, and the same direction (NT-v2 500M, layer 24, the
pooled HepG2 + K562 fit, no refit) gives P(bound | alt).

ΔP = P(bound | alt) − P(bound | ref): how much the mutation changes how bound the model thinks this site is, read in
its 600 bp context. No PWM and no MPRA label is used.

**Main score, fixed in advance:** ΣΔP = ΔP summed over all of the variant's motif occurrences (all FIMO hits of the
251 TFs that contain it, not only those motifbreakR calls). It also covers variants inside a motif that motifbreakR
does not call (about 82k of the 150k variants with a hit).

**Section 1 (GPU, ~45–60 min):** P(bound | alt) of every (variant, occurrence) pair, one file per TF in
`malinois/palt_nt500m/` (an interrupted session continues where it stopped).
**Section 2 (CPU, ~10–20 min):**

| | question |
|---|---|
| 2.2 | does ΣΔP predict Δ, compared with motifbreakR (one call / summed) and deltaSVM, on the variants motifbreakR calls, on those it does not, and on all with a hit |
| 2.3 | does ΣΔP add to summed motifbreakR, to deltaSVM, and to both (chromosome-half CV), per TF tier by CAV AUC; null N2 = ΔP shuffled among the pairs of the same TF |
| 2.4 | is ΣΔP's correlation with Δ larger in HepG2/K562 than in SK-N-SH (bootstrap 95% CI) |

deltaSVM comes from `Malinois MPRA 3` for whichever cells have been trained.

---
# Section 1 — P(bound | alt) *(GPU runtime)*

## 1.1 Setup and the readout, copied unchanged from the first Malinois notebook

In [ ]:
!pip install -q -U "transformers==4.57.6" "tokenizers==0.22.0" "huggingface_hub==0.34.0" "safetensors==0.4.5"
!pip install -q scikit-learn scipy pandas pyfaidx
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time, gc, sys, shutil
import numpy as np, pandas as pd
import torch
from scipy import stats
from pyfaidx import Fasta
from transformers import AutoTokenizer, AutoModel, AutoConfig, AutoModelForMaskedLM, AutoModelForCausalLM

C = json.load(open(f"{WORK_DIR}/config.json"))
POOL = f"{WORK_DIR}/fcav_pool"
MAL = f"{WORK_DIR}/malinois"
MWJ = json.load(open(f"{POOL}/mpra_windows.json"))
TAG, LAYER = "nt500m", 24
REGISTRY = {"nt500m": dict(hf="InstaDeepAI/nucleotide-transformer-v2-500m-multi-species", tok_map=("kmer", 6),
                           causal=False, batch=32)}
FIT_LAYERS = {"nt500m": [LAYER]}
BASE_WEIGHTED = False
BATCH = 32
WLEN, MOFF = C["window_len"], C["motif_offset"]
DEV = torch.device("cuda")
COMP = str.maketrans("ACGTN", "TGCAN")
rc = lambda s: s.translate(COMP)[::-1]
FIT_DIR = f"{POOL}/fits/{TAG}"
OUTA = f"{MAL}/palt_{TAG}"
os.makedirs(OUTA, exist_ok=True)

LOCAL_FA = "/content/hg38.fa"
if not os.path.exists(LOCAL_FA):
    print("copying hg38 to local disk (~3 GB, one-off)...")
    shutil.copy(C["genome_fa"], LOCAL_FA)
    if os.path.exists(C["genome_fa"] + ".fai"):
        shutil.copy(C["genome_fa"] + ".fai", LOCAL_FA + ".fai")
GENOME = Fasta(LOCAL_FA, sequence_always_upper=True)
V = pd.read_csv(f"{MAL}/variants.csv.gz", index_col="vi", dtype={"chrom": str}, low_memory=False,
                usecols=["vi", "chrom38", "pos38", "ref", "alt"])
H = pd.read_csv(f"{MAL}/hits.csv")
HP = H[~H["leak"]].drop_duplicates(["vi", "uid"]).reset_index(drop=True)
print(f"{len(HP):,} (variant, occurrence) pairs to embed with the alternative base, {HP['motif'].nunique()} TFs")

In [ ]:
def load_tokenizer(tag):
    spec = REGISTRY[tag]
    tokz = AutoTokenizer.from_pretrained(spec["hf"], trust_remote_code=True)
    tokz.padding_side = "right"          # several of these default to left (HyenaDNA)
    if tokz.pad_token is None:
        tokz.pad_token = tokz.eos_token if tokz.eos_token is not None else tokz.unk_token
    return tokz


def token_bases(spec, n):
    # (n, 2) base interval of each sequence token, for the arithmetic maps
    kind = spec["tok_map"][0]
    if kind == "kmer":
        k = spec["tok_map"][1]
        return np.stack([np.arange(n) * k, (np.arange(n) + 1) * k], 1)
    if kind == "char":
        return np.stack([np.arange(n), np.arange(n) + 1], 1)
    raise RuntimeError("offsets are resolved by the tokenizer, not here")


def encode(tokz, spec, seqs):
    # input_ids (B, T), attention mask (B, T), and per sequence: the positions of its sequence tokens in input_ids
    # and the base interval each one covers
    kind = spec["tok_map"][0]
    if kind == "offsets":
        assert tokz.is_fast, f"{spec['hf']} has no fast tokenizer, so offset mapping is unavailable"
        enc = tokz(seqs, return_tensors="pt", padding=True, return_offsets_mapping=True)
    else:
        enc = tokz(seqs, return_tensors="pt", padding=True)
    ids = enc["input_ids"]
    am = enc.get("attention_mask")
    if am is None:
        am = (ids != tokz.pad_token_id).long()
    amn = am.numpy()
    pos, spans = [], []
    if kind == "offsets":
        om = enc["offset_mapping"].numpy()
        for b in range(len(seqs)):
            keep = (om[b, :, 1] > om[b, :, 0]) & (amn[b] == 1)
            pos.append(np.flatnonzero(keep))
            spans.append(om[b, keep])
    else:
        for b in range(len(seqs)):
            sp = np.array(tokz.get_special_tokens_mask(ids[b].tolist(), already_has_special_tokens=True))
            p = np.flatnonzero((sp == 0) & (amn[b] == 1))
            pos.append(p)
            spans.append(token_bases(spec, len(p)))
        assert len({len(p) for p in pos}) == 1, f"{spec['hf']}: windows of equal length gave different token counts"
    return ids, am, pos, spans


def span_weights(bases, lo, hi):
    # per-token weight for the base interval [lo, hi)
    ov = np.minimum(bases[:, 1], hi) - np.maximum(bases[:, 0], lo)
    ov = np.clip(ov, 0, None).astype(np.float32)
    if not BASE_WEIGHTED:
        ov = (ov > 0).astype(np.float32)
    assert ov.sum() > 0, f"no token covers bases [{lo}, {hi})"
    return ov / ov.sum()


def load_model(tag):
    spec = REGISTRY[tag]
    # the class the checkpoint ships its own code for (NT-v2 has AutoModelForMaskedLM but no AutoModel)
    amap = getattr(AutoConfig.from_pretrained(spec["hf"], trust_remote_code=True), "auto_map", None) or {}
    cls = next((c for k, c in (("AutoModel", AutoModel), ("AutoModelForMaskedLM", AutoModelForMaskedLM),
                               ("AutoModelForCausalLM", AutoModelForCausalLM)) if k in amap), AutoModel)
    mdl = cls.from_pretrained(spec["hf"], trust_remote_code=True).to(DEV).eval()
    # DNABERT-2's triton flash-attention kernel does not compile on current triton; its code falls back to PyTorch
    for name in {type(m).__module__ for m in mdl.modules()}:
        mod = sys.modules.get(name)
        if mod is not None and getattr(mod, "flash_attn_qkvpacked_func", None) is not None:
            mod.flash_attn_qkvpacked_func = None
            print(f"{tag}: triton flash-attention disabled in {name.split('.')[-1]}; using PyTorch attention")
    if spec.get("out") == "dnabert2" and getattr(mdl, "pooler", None) is not None:
        mdl.pooler = None
    return mdl


POS, DEPTH, LOADED = {}, {}, {}


@torch.no_grad()
def hidden_stack(tag, model, ids, am):
    # one (layers, T_j, H) float tensor per sequence; only the layers in POS[tag] once prepare() has set it
    spec = REGISTRY[tag]
    if spec.get("out") == "dnabert2":
        # DNABERT-2 returns every encoder layer unpadded; one sequence at a time keeps the token order
        out = []
        for j in range(ids.shape[0]):
            t = ids[j][am[j].bool()][None].to(DEV)
            layers, _ = model(t, attention_mask=torch.ones_like(t), output_all_encoded_layers=True)
            S = torch.stack([x.reshape(t.shape[1], -1) for x in layers], 0).float()
            out.append(S[POS[tag]] if tag in POS else S)
        return out
    kw = {} if spec["causal"] else {"attention_mask": am.to(DEV)}
    hs = model(input_ids=ids.to(DEV), output_hidden_states=True, **kw).hidden_states
    if tag in POS:
        hs = [hs[i] for i in POS[tag]]
    S = torch.stack(hs, 1).float()
    return [S[j] for j in range(S.shape[0])]


PROBE = [h["window"] for h in MWJ["hits"][:4]]


def prepare(tag):
    # load the model; from one forward pass: number of layers, token tiling, and where FIT_LAYERS sit in the stack
    spec = REGISTRY[tag]
    mdl = load_model(tag)
    tokz = LOADED.get(tag, {}).get("tokz") or load_tokenizer(tag)
    LOADED[tag] = dict(spec=spec, tokz=tokz)
    POS.pop(tag, None)
    ids, am, pos, spans = encode(tokz, spec, PROBE[:2])
    S = hidden_stack(tag, mdl, ids, am)
    for j, Sj in enumerate(S):
        assert Sj.ndim == 3 and int(pos[j].max()) < Sj.shape[1]
        b = spans[j]
        assert b[0, 0] == 0 and b[-1, 1] == WLEN and (b[1:, 0] == b[:-1, 1]).all(), f"{tag}: tokens do not tile"
    nl = S[0].shape[0]
    lids = list(range(1, nl + 1)) if spec.get("out") == "dnabert2" else list(range(nl))   # DNABERT-2: no embedding layer
    DEPTH[tag] = lids[-1]
    miss = [l for l in FIT_LAYERS[tag] if l not in lids]
    assert not miss, f"{tag}: layers {miss} do not exist (ids {lids[0]}..{lids[-1]})"
    POS[tag] = [lids.index(l) for l in FIT_LAYERS[tag]]
    print(f"{tag}: {nl} layers (depth {DEPTH[tag]}), fitting {FIT_LAYERS[tag]}, hidden {S[0].shape[2]}, "
          f"{len(pos[0])} tokens for {WLEN} bp")
    del S
    return mdl


@torch.no_grad()
def readout(tag, model, seqs, lohi):
    # (n, layers, D): D = hidden, or 2 * hidden for a causal model (forward ‖ reverse complement)
    L = LOADED[tag]
    tokz, spec = L["tokz"], L["spec"]
    passes = [(seqs, lohi)]
    if spec["causal"]:
        passes.append(([rc(s) for s in seqs], [(WLEN - b, WLEN - a) for a, b in lohi]))
    bs = spec.get("batch", BATCH)
    parts = []
    for ss, lh in passes:
        out = None
        for i in range(0, len(ss), bs):
            chunk, lhc = ss[i:i + bs], lh[i:i + bs]
            ids, am, pos, spans = encode(tokz, spec, chunk)
            S = hidden_stack(tag, model, ids, am)
            if out is None:
                out = np.zeros((len(ss), S[0].shape[0], S[0].shape[2]), np.float16)
            for j in range(len(chunk)):
                w = torch.tensor(span_weights(spans[j], *lhc[j]), device=DEV)
                p = torch.tensor(pos[j], device=DEV)
                out[i + j] = (S[j][:, p] * w[None, :, None]).sum(1).half().cpu().numpy()
            del S
        parts.append(out)
    if len(parts) == 1:
        return parts[0]
    if spec.get("rc_mode", "concat") == "mean":
        return ((parts[0].astype(np.float32) + parts[1].astype(np.float32)) / 2).astype(np.float16)
    return np.concatenate(parts, 2)

## 1.2 Check: recomputing P(bound | ref) for a sample must give the stored P

The decision value is ((x − mu) / sd) · w + b with `w_all`, `mu`, `sd` of the TF's fit at layer 24, exactly as the
stored P(bound) of `malinois/p_nt500m/`.

In [ ]:
def direction(m):
    z = np.load(f"{FIT_DIR}/{m}.npz", allow_pickle=False)
    li = [int(l) for l in z["layers"]].index(LAYER)
    return z["w_all"][li].astype(np.float64), z["mu"][li].astype(np.float64), z["sd"][li].astype(np.float64)


def decision(m, windows, w_motif, mdl):
    w, mu, sd = direction(m)
    X = readout(TAG, mdl, windows, [(MOFF, MOFF + w_motif)] * len(windows))[:, 0].astype(np.float64)
    return ((X - mu) / sd) @ w[:-1] + w[-1]


def windows_of(g, alt):
    out = []
    for c, s, vi in zip(g["chrom38"].values, g["g38"].values, g["vi"].values):
        ws = int(s) - MOFF
        x = GENOME[c][ws:ws + WLEN].seq
        if alt:
            k = int(V.at[vi, "pos38"]) - ws
            assert 0 <= k < WLEN and x[k] == V.at[vi, "ref"], f"variant {vi}: reference base not at {k}"
            x = x[:k] + V.at[vi, "alt"] + x[k + 1:]
        assert len(x) == WLEN and "N" not in x
        out.append(x)
    return out


mdl = prepare(TAG)
PREF = pd.concat([pd.read_csv(f"{MAL}/p_{TAG}/{f}") for f in os.listdir(f"{MAL}/p_{TAG}")
                  if f.endswith(".csv") and not f.endswith(".part.csv")]).set_index("uid")["P"]
rows = []
for m in HP["motif"].value_counts().index[:5]:
    g = HP[HP["motif"] == m].drop_duplicates("uid").head(200)
    p = 1 / (1 + np.exp(-decision(m, windows_of(g, False), int(g["b"].iloc[0] - g["a"].iloc[0]), mdl)))
    rows.append({"TF": m, "n": len(g), "max |P recomputed - P stored|": np.abs(p - PREF.loc[g["uid"]].values).max()})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

## 1.3 Every (variant, occurrence) pair with the alternative base

In [ ]:
todo = [(m, g) for m, g in HP.groupby("motif") if not os.path.exists(f"{OUTA}/{m}.csv")]
n_left = sum(len(g) for _, g in todo)
print(f"{len(todo)} TFs, {n_left:,} pairs to do")
t_all, done = time.time(), 0
for k, (m, g) in enumerate(todo):
    d = decision(m, windows_of(g, True), int(g["b"].iloc[0] - g["a"].iloc[0]), mdl)
    pd.DataFrame({"vi": g["vi"].values, "uid": g["uid"].values, "dec_alt": d}).to_csv(f"{OUTA}/{m}.part.csv", index=False)
    os.replace(f"{OUTA}/{m}.part.csv", f"{OUTA}/{m}.csv")
    done += len(g)
    el = time.time() - t_all
    print(f"[{k + 1}/{len(todo)}] {m:10s} {len(g):6,} pairs  [{el / 60:.1f} min; ~{el / done * (n_left - done) / 60:.0f} min left]", flush=True)
del mdl; gc.collect(); torch.cuda.empty_cache()

In [ ]:
from google.colab import runtime
runtime.unassign()

---
# Section 2 — evaluation *(CPU runtime)*

## 2.1 Load

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, time
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score

MAL = f"{WORK_DIR}/malinois"
POOL = f"{WORK_DIR}/fcav_pool"
OUT = f"{MAL}/eval_deltaP"
os.makedirs(OUT, exist_ok=True)
CELLS3 = ["HepG2", "K562", "SKNSH"]
TIERS = [("all TFs", 0.0), ("AUC >= 0.70", 0.70), ("AUC >= 0.75", 0.75), ("AUC >= 0.80", 0.80)]
ACTIVE_MIN, Z_CLEAR = 1.0, 2.0
N_PERM, N_BOOT, LAYER = 200, 300, 24
pd.set_option("display.width", 250)
fmt = lambda x: f"{x:+.3f}"

V = pd.read_csv(f"{MAL}/variants.csv.gz", index_col="vi", dtype={"chrom": str}, low_memory=False)
DS_CELLS = [c for c in CELLS3 if os.path.exists(f"{MAL}/deltasvm/{c}.variants.csv")]
for c in DS_CELLS:
    V[f"ds_{c}"] = pd.read_csv(f"{MAL}/deltasvm/{c}.variants.csv", index_col="vi")["dsvm"].reindex(V.index)
print("deltaSVM available for:", DS_CELLS)
H = pd.read_csv(f"{MAL}/hits.csv")
HP = H[~H["leak"]].drop_duplicates(["vi", "uid"])[["vi", "uid", "motif"]]
rd = lambda d: pd.concat([pd.read_csv(f"{d}/{f}") for f in os.listdir(d) if f.endswith(".csv") and not f.endswith(".part.csv")])
PREF = rd(f"{MAL}/p_nt500m").set_index("uid")["P"]
PALT = rd(f"{MAL}/palt_nt500m")
HP = HP.merge(PALT, on=["vi", "uid"])
HP["P_ref"] = PREF.reindex(HP["uid"]).values
HP["P_alt"] = 1 / (1 + np.exp(-HP["dec_alt"]))
HP["dP"] = HP["P_alt"] - HP["P_ref"]
HP["dL"] = HP["dec_alt"] - np.log(np.clip(HP["P_ref"], 1e-12, 1) / np.clip(1 - HP["P_ref"], 1e-12, 1))
CALLS = pd.read_csv(f"{MAL}/calls.csv")
HP = HP.merge(CALLS[["vi", "motif"]].assign(called=True), on=["vi", "motif"], how="left")
HP["called"] = HP["called"].fillna(False).astype(bool)
AUC = {}
for m in HP["motif"].unique():
    z = np.load(f"{POOL}/fits/nt500m/{m}.npz", allow_pickle=False)
    AUC[m] = float(np.asarray(z["auc"])[[int(l) for l in z["layers"]].index(LAYER)])
HP["auc"] = HP["motif"].map(AUC)
DPD = f"{MAL}/deltaP"
os.makedirs(DPD, exist_ok=True)
HP[["vi", "uid", "motif", "called", "auc", "P_ref", "P_alt", "dP", "dL"]].to_csv(f"{DPD}/pairs.csv.gz", index=False)
SV = HP.groupby("vi").agg(**{"sum_dP": ("dP", "sum"), "sum_dP_calls": ("dP", lambda x: x[HP.loc[x.index, "called"]].sum()),
                             "sum_dlogit": ("dL", "sum"), "n_occurrences": ("dP", "size")})
SV.to_csv(f"{DPD}/variants.csv", index_label="vi")
print(f"saved ΔP per pair and ΣΔP per variant to {DPD}/")
print(f"{len(HP):,} (variant, occurrence) pairs with ΔP on {HP['vi'].nunique():,} variants; "
      f"{int(HP['called'].sum()):,} of them are motifbreakR calls")
print(HP["dP"].describe().to_string(float_format=lambda x: f"{x:+.4f}"))

## 2.2 Does ΣΔP predict Δ? Next to motifbreakR and deltaSVM (no MPRA training in any of them)

Three groups of variants with at least one motif occurrence: those motifbreakR calls, those it does not call (where
motifbreakR predicts 0), and all of them. Pearson with Δ and AUROC for clear effects (|Δ| ≥ 2 SE).

In [ ]:
def pear(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a - a.mean(), b - b.mean()
    den = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / den) if den > 0 else np.nan


def auroc(lab, s):
    return roc_auc_score(lab, s) if 0 < lab.sum() < len(lab) else np.nan


def context(L):
    L = L.sort_values(["vi", "motif"]).reset_index(drop=True)
    VI = np.unique(L["vi"].values)
    X = dict(VI=VI, cv=np.searchsorted(VI, L["vi"].values), cm=pd.factorize(L["motif"])[0], dP=L["dP"].values,
             dL=L["dL"].values, called=L["called"].values)
    X["base"] = np.argsort(X["cm"], kind="stable")
    tfs = set(L["motif"])
    Cc = CALLS[CALLS["motif"].isin(tfs) & CALLS["vi"].isin(VI)]
    n = len(VI)
    ci = np.searchsorted(VI, Cc["vi"].values)
    X["SUM_AD"] = np.bincount(ci, Cc["alleleDiff"].values, minlength=n)
    pk = Cc.loc[Cc["alleleDiff"].abs().sort_values(ascending=False, kind="stable").index].drop_duplicates("vi")
    X["PICK_AD"] = np.zeros(n); X["PICK_AD"][np.searchsorted(VI, pk["vi"].values)] = pk["alleleDiff"].values
    X["HAS_CALL"] = np.zeros(n, bool); X["HAS_CALL"][ci] = True
    chrom = V.loc[VI, "chrom"].astype(str).values
    X["FOLD"] = np.array([int(ch) % 2 if ch.isdigit() else 0 for ch in chrom])
    X["Y"] = {c: V.loc[VI, f"d_{c}"].values for c in CELLS3}
    X["CL"] = {c: np.abs(X["Y"][c]) / V.loc[VI, f"se_{c}"].values >= Z_CLEAR for c in CELLS3}
    X["DS"] = {c: V.loc[VI, f"ds_{c}"].values for c in DS_CELLS}
    X["SETS"] = {(c, s): np.isfinite(X["Y"][c]) & ((V.loc[VI, f"act_{c}"].values >= ACTIVE_MIN) if s == "active" else True)
                 for c in CELLS3 for s in ("all", "active")}
    return X


def sums(X, dP):
    n = len(X["VI"])
    return {"ΣΔP": np.bincount(X["cv"], dP, minlength=n), "ΣΔP, calls only": np.bincount(X["cv"], dP * X["called"], minlength=n)}


X0 = context(HP)
S0 = sums(X0, X0["dP"])
S0["ΣΔlogit"] = np.bincount(X0["cv"], X0["dL"], minlength=len(X0["VI"]))
rows = []
for (c, s), m in X0["SETS"].items():
    for grp, gm in (("called by motifbreakR", X0["HAS_CALL"]), ("not called", ~X0["HAS_CALL"]), ("all with a motif hit", np.ones(len(m), bool))):
        k = m & gm
        y, cl = X0["Y"][c][k], X0["CL"][c][k]
        preds = {"motifbreakR": X0["PICK_AD"][k], "summed motifbreakR": X0["SUM_AD"][k], **{nm: v[k] for nm, v in S0.items()}}
        if c in DS_CELLS:
            preds["deltaSVM"] = X0["DS"][c][k]
        row = {"cell": c, "variants": s, "group": grp, "n": int(k.sum())}
        for nm, pr in preds.items():
            row[f"Pearson {nm}"] = pear(pr, y)
        for nm, pr in preds.items():
            row[f"AUROC {nm}"] = auroc(cl, np.abs(pr))
        rows.append(row)
T2 = pd.DataFrame(rows).set_index(["cell", "variants", "group"])
T2.to_csv(f"{OUT}/levels.csv")
print("=== Pearson with Δ\n")
print(T2[[c for c in T2.columns if c.startswith("Pearson")]].to_string(float_format=fmt))
print("\n=== AUROC for clear effects\n")
print(T2[[c for c in T2.columns if c.startswith("AUROC")]].to_string(float_format=fmt))

## 2.3 Does ΣΔP add to summed motifbreakR and to deltaSVM? Every TF tier

On all variants with a motif hit of the tier's TFs. Linear fit on one chromosome half, scored on the other;
gain = out-of-sample Pearson with ΣΔP minus without. p N2 = share of 200 shuffles of ΔP among the pairs of the same
TF with at least this gain (the whole computation redone). Rows with deltaSVM only for the cells it has been trained in.

In [ ]:
def cv_pear(cols, y, f):
    Z = np.column_stack([np.ones(len(y))] + cols)
    out = np.zeros(len(y))
    for k in (0, 1):
        tr, te = f != k, f == k
        out[te] = Z[te] @ np.linalg.lstsq(Z[tr], y[tr], rcond=None)[0]
    return pear(out, y)


def shuffle_tf(X, rng):
    o = np.lexsort((rng.random(len(X["dP"])), X["cm"]))
    out = np.empty_like(X["dP"]); out[X["base"]] = X["dP"][o]
    return out


BASES = {"summed motifbreakR": lambda X, c, m: [X["SUM_AD"][m]],
         "deltaSVM": lambda X, c, m: [X["DS"][c][m]],
         "deltaSVM + summed motifbreakR": lambda X, c, m: [X["DS"][c][m], X["SUM_AD"][m]]}


def gains(X, dP):
    sP = sums(X, dP)["ΣΔP"]
    out = {}
    for (c, s), m in X["SETS"].items():
        y, f = X["Y"][c][m], X["FOLD"][m]
        out[c, s, "ΣΔP alone"] = (pear(sP[m], y), np.nan)
        for b, fn in BASES.items():
            if b.startswith("deltaSVM") and c not in DS_CELLS:
                continue
            base = fn(X, c, m)
            r0 = cv_pear(base, y, f)
            out[c, s, b] = (r0, cv_pear(base + [sP[m]], y, f) - r0)
    return out


ROWS = []
t0 = time.time()
for name, thr in TIERS:
    X = context(HP[HP["auc"] >= thr])
    rng = np.random.default_rng(7)
    obs = gains(X, X["dP"])
    null = [gains(X, shuffle_tf(X, rng)) for _ in range(N_PERM)]
    for key, (lev, g) in obs.items():
        nl = np.array([n_[key] for n_ in null])
        c, s, b = key
        if b == "ΣΔP alone":
            ROWS.append({"threshold": name, "cell": c, "variants": s, "base": b, "n": int(X["SETS"][c, s].sum()),
                         "Pearson base (CV)": np.nan, "gain with ΣΔP": lev, "p N2": np.mean(nl[:, 0] >= lev)})
        else:
            ROWS.append({"threshold": name, "cell": c, "variants": s, "base": b, "n": int(X["SETS"][c, s].sum()),
                         "Pearson base (CV)": lev, "gain with ΣΔP": g, "p N2": np.mean(nl[:, 1] >= g)})
    print(f"{name}: {len(np.unique(X['cm']))} TFs, {len(X['VI']):,} variants  [{(time.time() - t0) / 60:.1f} min]", flush=True)
R = pd.DataFrame(ROWS)
R["_t"] = R["threshold"].map({t: i for i, (t, _) in enumerate(TIERS)})
R = R.sort_values(["_t", "base"], kind="stable").drop(columns="_t").set_index(["threshold", "base", "cell", "variants"])
R.to_csv(f"{OUT}/gains.csv")
print("\n('ΣΔP alone' rows: 'gain with ΣΔP' is ΣΔP's own Pearson with Δ, and p N2 tests it against the shuffles)\n")
print(R.to_string(float_format=fmt))

## 2.4 Cell contrast: is ΣΔP's correlation with Δ larger in HepG2/K562 than in SK-N-SH?

The CAV was trained on HepG2 + K562 ChIP. Same variants in every cell (all with a motif hit); bootstrap 95% CI.

In [ ]:
CROWS = []
for name, thr in TIERS:
    X = context(HP[HP["auc"] >= thr])
    sP = sums(X, X["dP"])["ΣΔP"]
    rng = np.random.default_rng(11)
    for s in ("all", "active"):
        for c in ("HepG2", "K562"):
            u = np.flatnonzero(X["SETS"][c, s] | X["SETS"]["SKNSH", s])
            def diff(j):
                a, b = j[X["SETS"][c, s][j]], j[X["SETS"]["SKNSH", s][j]]
                return pear(sP[a], X["Y"][c][a]) - pear(sP[b], X["Y"]["SKNSH"][b])
            d = [diff(rng.choice(u, len(u))) for _ in range(N_BOOT)]
            CROWS.append({"threshold": name, "contrast": f"{c} - SK-N-SH", "variants": s, "difference in Pearson(ΣΔP, Δ)": diff(u),
                          "95% CI low": np.percentile(d, 2.5), "95% CI high": np.percentile(d, 97.5)})
CR = pd.DataFrame(CROWS).set_index(["threshold", "contrast", "variants"])
CR.to_csv(f"{OUT}/contrast.csv")
print(CR.to_string(float_format=fmt))

## End

Outputs in `malinois/eval_deltaP/`: `levels.csv`, `gains.csv`, `contrast.csv`; P(bound | alt) in `malinois/palt_nt500m/`;
ΔP per (variant, occurrence) in `malinois/deltaP/pairs.csv.gz` and ΣΔP per variant in `malinois/deltaP/variants.csv`.